#### Decoding Yelp: A Deep Dive into “Spark”-ling Customer Reviews and Business Trends

#### Sentiment Analysis - TextBlob

#### Scaling Analysis

# Full csv data (no Spark and HDFS) 

In [7]:
%%time
data = pd.read_csv('df_import.csv')

CPU times: user 48.6 s, sys: 5.6 s, total: 54.1 s
Wall time: 1min 7s


In [15]:
def get_sentiment(text):
    # Create a TextBlob object
    blob = TextBlob(text)
    # Get the polarity of the text
    polarity = blob.sentiment.polarity
    # Determine sentiment based on polarity
    if polarity > 0.2:
        return 'positive'
    else:
        return 'negative'

In [10]:
%%time
data['sentiment_full'] = data['text'].apply(get_sentiment)

CPU times: user 1h 25min 43s, sys: 822 ms, total: 1h 25min 44s
Wall time: 1h 25min 45s


CPU times: user 1h 25min 43s, sys: 822 ms, total: 1h 25min 44s

Wall time: 1h 25min 45s

In [ ]:
data[0:5]

# Strong Scaling - with Spark and HDFS

In [1]:
import findspark
findspark.init('/home/ubuntu/spark-3.3.1-bin-hadoop3')
findspark.find()

'/home/ubuntu/spark-3.3.1-bin-hadoop3'

In [2]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from pyspark.streaming import StreamingContext
from textblob import TextBlob

In [3]:
from pyspark.sql import SparkSession

# The entry point into all functionality in Spark is the SparkSession class.
spark = (SparkSession
	.builder
	.appName("DS5110/CS5501: Yelp")
	.master("spark://172.31.90.235:7077")
	.config("spark.executor.memory", "1024M")
	.getOrCreate())


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


24/04/22 16:07:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
import nltk
#nltk.download('punkt')
#nltk.download('averaged_perceptron_tagger')
#nltk.download('brown')

In [5]:
import pandas as pd
import random
import time

In [6]:
def get_sentiment(text):
    # Create a TextBlob object
    blob = TextBlob(text)
    # Get the polarity of the text
    polarity = blob.sentiment.polarity
    # Determine sentiment based on polarity
    if polarity > 0.2:
        return 'positive'
    else:
        return 'negative'

In [7]:
sentiment = udf(get_sentiment,StringType())

In [8]:
sentiment

<function __main__.get_sentiment(text)>

## 3 workers, 6 CPUs

In [6]:
%%time
# Read the CSV with options to handle quotes and multiline rows
data = spark.read.option("header", "true") \
               .option("quote", "\"") \
               .option("escape", "\"") \
               .option("multiline", "true") \
               .csv("hdfs://172.31.90.235:9000/df_import.csv")

data.show()


+---+--------------------+---------+
|_c0|                text|sentiment|
+---+--------------------+---------+
|  0|If you decide to ...| negative|
|  1|This is the secon...| negative|
|  2|The place is cute...| positive|
|  3|We came on a Satu...| negative|
|  4|Mediocre at best....| negative|
|  5|When I was shown ...| negative|
|  6|Food is fantastic...| positive|
|  7|I went on a Thurs...| positive|
|  8|Update: I deducte...| negative|
|  9|The food better b...| negative|
| 10|This place is con...| negative|
| 11|"Who's in the kit...| negative|
| 12|Hopefully they wi...| negative|
| 13|Not Impressed at ...| negative|
| 14|never coming back...| negative|
| 15|Went here over th...| positive|
| 16|This place is del...| positive|
| 17|I don't recommend...| negative|
| 18|Well, lots to say...| negative|
| 19|So we live turnin...| negative|
+---+--------------------+---------+
only showing top 20 rows

CPU times: user 9.57 ms, sys: 5.34 ms, total: 14.9 ms
Wall time: 16.5 s


CPU times: user 9.57 ms, sys: 5.34 ms, total: 14.9 ms

Wall time: 16.5 s

In [10]:
%%time
# Perform sentiment analysis with Spark 
data_sent = data.withColumn('sentiment_one', sentiment(data['text'])).count()

CPU times: user 12.2 ms, sys: 4.58 ms, total: 16.8 ms
Wall time: 32.5 s


CPU times: user 12.2 ms, sys: 4.58 ms, total: 16.8 ms

Wall time: 32.5 s


In [ ]:
%%time
#data_sent.show(10000)
data.withColumn('sentiment_one', sentiment(data['text'])).show(10000)

CPU times: user 24.4 ms, sys: 197 µs, total: 24.6 ms

Wall time: 29.1 s

In [12]:
# Stop the Spark session
#spark.stop()

## 2 workers, 4 CPUs

In [6]:
%%time
# Read the CSV with options to handle quotes and multiline rows
data = spark.read.option("header", "true") \
               .option("quote", "\"") \
               .option("escape", "\"") \
               .option("multiline", "true") \
               .csv("hdfs://172.31.90.235:9000/df_import.csv")

data.show()


+---+--------------------+---------+
|_c0|                text|sentiment|
+---+--------------------+---------+
|  0|If you decide to ...| negative|
|  1|This is the secon...| negative|
|  2|The place is cute...| positive|
|  3|We came on a Satu...| negative|
|  4|Mediocre at best....| negative|
|  5|When I was shown ...| negative|
|  6|Food is fantastic...| positive|
|  7|I went on a Thurs...| positive|
|  8|Update: I deducte...| negative|
|  9|The food better b...| negative|
| 10|This place is con...| negative|
| 11|"Who's in the kit...| negative|
| 12|Hopefully they wi...| negative|
| 13|Not Impressed at ...| negative|
| 14|never coming back...| negative|
| 15|Went here over th...| positive|
| 16|This place is del...| positive|
| 17|I don't recommend...| negative|
| 18|Well, lots to say...| negative|
| 19|So we live turnin...| negative|
+---+--------------------+---------+
only showing top 20 rows

CPU times: user 12.6 ms, sys: 714 µs, total: 13.3 ms
Wall time: 13.8 s


CPU times: user 12.6 ms, sys: 714 µs, total: 13.3 ms

Wall time: 13.8 s

In [11]:
%%time
# Perform sentiment analysis with Spark 
data_sent = data.withColumn('sentiment_two', sentiment(data['text'])).count()

CPU times: user 18.8 ms, sys: 1.97 ms, total: 20.7 ms
Wall time: 34.6 s


CPU times: user 18.8 ms, sys: 1.97 ms, total: 20.7 ms

Wall time: 34.6 s

In [ ]:
%%time
#data_sent.show(10000)
data.withColumn('sentiment_two', sentiment(data['text'])).show(10000)

CPU times: user 13.3 ms, sys: 9.51 ms, total: 22.8 ms

Wall time: 30.8 s

In [12]:
# Stop the Spark session
#spark.stop()

## 1 workers, 2 CPUs

In [9]:
%%time
# Read the CSV with options to handle quotes and multiline rows
data = spark.read.option("header", "true") \
               .option("quote", "\"") \
               .option("escape", "\"") \
               .option("multiline", "true") \
               .csv("hdfs://172.31.90.235:9000/df_import.csv")

data.show()

+---+--------------------+---------+
|_c0|                text|sentiment|
+---+--------------------+---------+
|  0|If you decide to ...| negative|
|  1|This is the secon...| negative|
|  2|The place is cute...| positive|
|  3|We came on a Satu...| negative|
|  4|Mediocre at best....| negative|
|  5|When I was shown ...| negative|
|  6|Food is fantastic...| positive|
|  7|I went on a Thurs...| positive|
|  8|Update: I deducte...| negative|
|  9|The food better b...| negative|
| 10|This place is con...| negative|
| 11|"Who's in the kit...| negative|
| 12|Hopefully they wi...| negative|
| 13|Not Impressed at ...| negative|
| 14|never coming back...| negative|
| 15|Went here over th...| positive|
| 16|This place is del...| positive|
| 17|I don't recommend...| negative|
| 18|Well, lots to say...| negative|
| 19|So we live turnin...| negative|
+---+--------------------+---------+
only showing top 20 rows

CPU times: user 11.6 ms, sys: 636 µs, total: 12.2 ms
Wall time: 12.2 s


CPU times: user 11.3 ms, sys: 1.78 ms, total: 13.1 ms

Wall time: 12 s

In [10]:
%%time
# Perform sentiment analysis with Spark 
data_sent = data.withColumn('sentiment_three', sentiment(data['text'])).count()

CPU times: user 17.7 ms, sys: 494 µs, total: 18.2 ms
Wall time: 31.9 s


CPU times: user 17.7 ms, sys: 494 µs, total: 18.2 ms

Wall time: 31.9 s

In [ ]:
%%time
data.withColumn('sentiment_three', sentiment(data['text'])).show(10000)

CPU times: user 17.2 ms, sys: 6.08 ms, total: 23.3 ms

Wall time: 29 s

In [ ]:
# Stop the Spark session
#spark.stop()

# Weak Scaling - with Spark and HDFS

## 5000 observations with 1 worker

In [ ]:
%%time
data.withColumn('sentiment_three', sentiment(data['text'])).show(5000)

CPU times: user 23.9 ms, sys: 2.48 ms, total: 26.4 ms

Wall time: 29.3 s

## 10 000 observations with 2 workers

In [ ]:
%%time
data.withColumn('sentiment_two', sentiment(data['text'])).show(10000)

CPU times: user 13.3 ms, sys: 9.51 ms, total: 22.8 ms

Wall time: 30.8 s